# 🚀 GoViralIQ
## Notebook 1: Data Exploration & Feature Engineering
**Project:** Predicting Instagram Engagement & Auditing Algorithmic Fairness Across Creator Niches  
**Author:** Chastity Lewis  
**Course:** CISC 540 — Computational Data Analysis | Mercy University | Spring 2026  

---

### 📌 Notebook Goals
In this notebook we will:
1. Load and inspect the raw Kaggle dataset
2. Filter for Instagram posts only
3. Explore the shape, data types, and distributions
4. Check for missing values and duplicates
5. Engineer the `engagement_rate` feature
6. Engineer the binary `viral` target variable
7. Save the prepared dataset for Notebook 2

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#6B21A8', '#D8B4FE', '#1F1F1F', '#F3E8FF', '#9333EA']

print('✅ Libraries loaded!')

---
## Step 2: Load the Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'✅ Loaded: {filename}')
print(f'📊 Raw Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

---
## Step 3: Filter for Instagram Only

In [ ]:
df = df_raw[df_raw['Platform'] == 'Instagram'].copy()
df = df.reset_index(drop=True)

print(f'📱 Platforms in raw data: {df_raw["Platform"].value_counts().to_dict()}')
print(f'✅ After filtering to Instagram: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Step 4: Inspect the Data

In [ ]:
print('📋 Column Names & Data Types:')
print(df.dtypes)
print(f'\n📐 Shape: {df.shape}')

In [ ]:
print('🔍 First 5 rows:')
df.head()

In [ ]:
print('📊 Descriptive Statistics (Numerical Columns):')
df.describe()

---
## Step 5: Check Missing Values & Duplicates

In [ ]:
print('❓ Missing Values:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0 else '✅ No missing values found!')

print(f'\n🔁 Duplicate Rows: {df.duplicated().sum()}')

---
## Step 6: Explore Categorical Variables

In [ ]:
print('📂 Content Type Distribution:')
print(df['Content_Type'].value_counts())
print()
print('🌍 Region Distribution:')
print(df['Region'].value_counts())
print()
print('📈 Engagement Level Distribution:')
print(df['Engagement_Level'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Content Type
ct_counts = df['Content_Type'].value_counts()
axes[0].bar(ct_counts.index, ct_counts.values, color=COLORS[0], edgecolor='white')
axes[0].set_title('Posts by Content Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Content Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Region
region_counts = df['Region'].value_counts()
axes[1].bar(region_counts.index, region_counts.values, color=COLORS[1], edgecolor='white')
axes[1].set_title('Posts by Region', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Engagement Level
el_counts = df['Engagement_Level'].value_counts()
axes[2].bar(el_counts.index, el_counts.values, color=COLORS[4], edgecolor='white')
axes[2].set_title('Posts by Engagement Level', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Engagement Level')
axes[2].set_ylabel('Count')

plt.suptitle('GoViralIQ — Categorical Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('✅ Categorical distributions plotted!')

---
## Step 7: Explore Numerical Variables

In [ ]:
num_cols = ['Views', 'Likes', 'Shares', 'Comments']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color=COLORS[0], edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=2,
                    label=f'Mean: {df[col].mean():,.0f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', linewidth=2,
                    label=f'Median: {df[col].median():,.0f}')
    axes[i].set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=9)

plt.suptitle('GoViralIQ — Numerical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Numerical distributions plotted!')

---
## Step 8: Engineer `engagement_rate`

In [ ]:
# engagement_rate = (Likes + Comments + Shares) / Views
# Replace 0 Views with 1 to avoid division by zero
df['engagement_rate'] = (
    (df['Likes'] + df['Comments'] + df['Shares']) /
    df['Views'].replace(0, 1)
)

print('✅ engagement_rate engineered!')
print(f'\n📊 Engagement Rate Summary:')
print(f'  Mean:    {df["engagement_rate"].mean():.4f}')
print(f'  Median:  {df["engagement_rate"].median():.4f}')
print(f'  Std Dev: {df["engagement_rate"].std():.4f}')
print(f'  Min:     {df["engagement_rate"].min():.4f}')
print(f'  Max:     {df["engagement_rate"].max():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['engagement_rate'], bins=50, color=COLORS[0], edgecolor='white', alpha=0.85)
axes[0].axvline(df['engagement_rate'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {df["engagement_rate"].mean():.2f}')
axes[0].axvline(df['engagement_rate'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {df["engagement_rate"].median():.2f}')
axes[0].set_title('Engagement Rate Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Engagement Rate')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot by Content Type
content_types = df['Content_Type'].unique()
data_by_type = [df[df['Content_Type'] == ct]['engagement_rate'].values for ct in content_types]
bp = axes[1].boxplot(data_by_type, labels=content_types, patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS * 2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Engagement Rate by Content Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Content Type')
axes[1].set_ylabel('Engagement Rate')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('GoViralIQ — Engagement Rate Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Engagement rate visualized!')

---
## Step 9: Engineer Binary `viral` Target

In [ ]:
# Define viral as engagement_rate >= 80th percentile
threshold = df['engagement_rate'].quantile(0.80)
df['viral'] = (df['engagement_rate'] >= threshold).astype(int)

viral_count = df['viral'].sum()
total = len(df)

print(f'✅ viral target engineered!')
print(f'\n📊 Viral Label Summary:')
print(f'  80th Percentile Threshold: {threshold:.4f}')
print(f'  Viral Posts (1):     {viral_count:,} ({viral_count/total*100:.1f}%)')
print(f'  Non-Viral Posts (0): {total - viral_count:,} ({(total-viral_count)/total*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Viral vs Non-Viral bar chart
viral_counts = df['viral'].value_counts().sort_index()
bars = axes[0].bar(['Non-Viral (0)', 'Viral (1)'], viral_counts.values,
                   color=[COLORS[1], COLORS[0]], edgecolor='white', width=0.5)
for bar, count in zip(bars, viral_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_title('Viral vs Non-Viral Posts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Viral rate by Content Type
viral_by_type = df.groupby('Content_Type')['viral'].mean() * 100
viral_by_type = viral_by_type.sort_values(ascending=False)
axes[1].bar(viral_by_type.index, viral_by_type.values, color=COLORS[0], edgecolor='white', alpha=0.85)
axes[1].axhline(y=20, color='red', linestyle='--', linewidth=1.5, label='Overall Average (20%)')
axes[1].set_title('Viral Rate by Content Type (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Content Type')
axes[1].set_ylabel('Viral Rate (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.suptitle('GoViralIQ — Viral Label Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Viral label visualized!')

---
## Step 10: Correlation Heatmap

In [ ]:
import matplotlib.colors as mcolors

num_df = df[['Views', 'Likes', 'Shares', 'Comments', 'engagement_rate', 'viral']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='RdPu', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(corr.columns, fontsize=11)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')

ax.set_title('GoViralIQ — Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print('✅ Correlation heatmap plotted!')

---
## Step 11: Dataset Summary Before Saving

In [ ]:
print('='*55)
print('       GoViralIQ — Dataset Summary (Notebook 1)')
print('='*55)
print(f'  Total Records:       {len(df):,}')
print(f'  Total Features:      {df.shape[1]}')
print(f'  Missing Values:      {df.isnull().sum().sum()}')
print(f'  Duplicate Rows:      {df.duplicated().sum()}')
print(f'  Viral Posts:         {df["viral"].sum():,} ({df["viral"].mean()*100:.1f}%)')
print(f'  Non-Viral Posts:     {(df["viral"]==0).sum():,} ({(df["viral"]==0).mean()*100:.1f}%)')
print(f'  Viral Threshold:     {threshold:.4f}')
print(f'  Avg Engagement Rate: {df["engagement_rate"].mean():.4f}')
print('='*55)
print()
print('📋 Final Columns:')
for col in df.columns:
    print(f'   • {col} ({df[col].dtype})')

---
## Step 12: Save Dataset for Notebook 2

In [ ]:
output_filename = 'goviraliq_instagram_raw.csv'
df.to_csv(output_filename, index=False)

# Download the file from Colab
files.download(output_filename)

print(f'✅ Dataset saved as: {output_filename}')
print(f'📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n➡️  Upload this file to Notebook 2 to continue the pipeline.')